---
title: 'Classification of uncertainty'
---

Before continuing the discussion how to quantify the uncertainty, let's start by having a look of different types of uncertainty. Depending on the domain, there are different ways of classifying uncertainty, which are only partly overlapping. Here, I will discuss two types of classifications. In many natural sciences, e.g. physics, one usually talks about **statistical/stochastic** and **systematic** uncertainty. In computer science one more often refer to **aleatoric** and **epistemic** uncertainty. These two classification schemes are often described as indentical making statistical---aleatoric and systematic---epistemic synonyms. In reality there are however some subtle differences, since the nature of uncertainty in a physical measurement and in a mathematical model are different.

## Statistical/stochastic uncertainty
If we repeat a measurement, say the voltage across a light bulb, several times we will not get the same value every time (provided the resolution of the voltmeter is sufficently good) since the voltage can never be absolutely stable. In the same way, if we measure the time interval between radioactive decays from some source, there will be some variation since the decay process is fundamentally stochastic. The same is true for any physical measurement - if you try to make it sufficiently precise, you will necessarily observe some fluctuations in the value you measure. The statistical uncertainty is relatively easy to handle: by making repeated measurements and averaging the result one find a better estimate of the true value than what a single measurement will give. Also, if one has some knowledge of the underlying statistical model for the fluctuations (Gaussian, Poisson, ...) there are concrete receipts on how to quantify the uncertainty of this estimate. Furthermore, if one calculates a quantity which is a function of two or more measured values, the uncertainty of the compund quantity can easily(ish) be calculated as well, as described in the previous section.

The statistical uncertainty can always be reduced by adding more data---the magnitude of the uncertainty scales as $\frac{1}{\sqrt{N}}$ where $N$ is number of measurements. However, given it's random nature one cannot reduce the impact of statistical uncertainty beyond a certain point by processing the data in a more clever way. For this reason, it is often referred to as an irreducible uncertainty.

## Systematic uncertainty
Systematic uncertainties relates to specific measurement apparatuses or procedures, and countrary to the statistical uncertainty will typically shift all measurement results in one direction. A trivial example may be a measurement rod which expands or contracts when the temperature changes. All measurements taken at the same temperatur will then be shifted by the same amount. There is no general procedure for how to estimate the systematic uncertainty - it depends too much on the nature of the measurement that is done. In many cases the best one can do is to perform the same measurement with a different apparatus/procedure and use the deviation as an estimate of the uncertainty. Also, there is no general prescription for how to combine the statistical uncertainty into a compound uncertainty when calculating a value based on several measured parameters. It is quite common to use the same procedure as for statistical uncertainties, but there is no statistical foundation for doing this.

Increasing the amount of data will in general not reduce the size of the systematic uncertainty. However, having more data at your disposal may in some cases allow you to improve measurements that again can let you reduce the systematic uncertainty.

## Aleatoric uncertainty
In the context of ML, aleatoric uncertainty is the uncertainty on the data which are used as input to the ML model. Making the connection to the statistical---systematic split of uncertainty, all statistical uncertainty is aleatoric, but not necessarily all systematic uncertainty. In general, there will be a systematic uncertainty connected to the measurement of the input values---and this $will$ contribute to the aleatoric uncertainty. The choise of model, an specific choises of hyperparameters will also contribute a systematic uncertainty, but this should be regarded as part of the epistemic uncertainty which is discussed in the next paragraph.

## Epistemic uncertainty
Epistemic uncertainty is in some sense also a systematic uncertainty, but usually referes only to uncertainties relating to the model which is used to analyse the data.

:::{.callout-tip title="Example"}
In the example from the previous section, the expression relating $T$, $L$ and $g$, $T = 2\pi\sqrt{\frac{L}{g}}$, is not quite precise but is only an approximation to a much more complicated expression. First of all, the expresision is derived assuming that all of the mass in the pendulum is located in a point at the end of the string. This is of course not true (but may be an ok approximation in some cases). Secondly, the derivation uses $\sin\theta \simeq \theta$ which only is true in the limit $\theta\to 0$. As is typical for an epistemic uncertainty, it can be reduced^[But it cannot be completely removed since the equation describing the pendulum doesn't have an analytic solution. This is no problem, however, since there will allways be statistical errors present as well. When the epistemic uncertainty is reduced to be much smaller than the systematic error one doesn't gain anything from further reducing it.] by improving the mathematical model that is used.
:::

When using a complex ML model like a deep NN, there is still the possibility that the model cannot adequately describe the relation we are trying to find. Due to the complexity--and opaqueness--of the ML model it is much harder in this case to evaluate how large of an epistemic uncertainty we have, and to find out how to reduce it if necessary. 

## Out-of-distribution data
If an instance which is not represented by the training data is fed through the trained ML model, most models will produce a prediction just as it would for any other instance. However, since this involves an extrapolation outside of domain from which we have information this inference will naturally come with an additional uncertainty on top of the usual set of uncertainties involved in the inference. This kind of uncertainty is often not discussed, but is important to include since it may be very large and difficult to quantify. In general one would expect the out-of-distribution uncertainty to grow with increasing distance to the training data, and as such a successful estimation of this uncertainty could be usable as an anomaly detector, flagging data not consistent with the training data set by assigning a particularly large uncertainty.

This source of uncertainty is often neglected in the discussion of uncertainty quantification, and according to [Everett2022]^[[Improving Out-of-Distribution Detection via Epistemic Uncertainty Adversarial Training](https://arxiv.org/abs/2209.03148), D Everett et al., arXiv:2209.03148 (2022)] common approaches to estimating the uncertainty of an ML model fail to yield reasonable estimates for this important case, particularly if the model is also required to give a very low false positive rate---which is often required for ML models that should be deployed into practical use.

For out-of-distribution data, there are two different cases which requires different handling. First there is "clean data", meaning that all the input features of the instance have plausible values, but (some of which) are out of the range represented in the training data. Say, we have an image classifier trying to determine bird species. If the classifier was never trained on images of gulls, it cannot be expected to recognize gull images even though the images are of perfect quality. The best we can hope for is that it returns 'no known class', or if this is not an option it should return some class, but with very large uncertainty. Secondly there is "adverserial data". This is an input instance specially designed to fool the classifier. Ideally, also this should come out as 'no known class' or some class, but with a large error.